# FINet industrial defect segmentation
Separate custom supervised MVTec AD experiment. Original FINet and ImageNet EfficientNet-B0 initialization; unchanged four-head structure loss. Boundary variant adds the existing differentiable loss at lambda 0.1.
Attach the private original MVTec input, the FINet industrial source input, and the latest saved notebook output when resuming. Use Save & Run All for a bounded server-side run. CC BY-NC-SA 4.0, academic non-commercial use. This is not the official normal-only anomaly benchmark.

In [ ]:
from pathlib import Path
import os, sys, json, shutil, subprocess, zipfile, hashlib
PROJECT = Path('/kaggle/working/boundary-aware-finet-industrial')
roots = [p.parent for p in Path('/kaggle/input').rglob('train_industrial.py')]
dedicated_roots = [p for p in roots if 'finet-industrial-source-november-13' in p.parts]
roots = dedicated_roots or roots
if len(roots) == 1:
    shutil.copytree(roots[0], PROJECT, dirs_exist_ok=True)
else:
    archives = list(Path('/kaggle/input').rglob('finet_industrial_source.zip'))
    assert len(archives) == 1, 'Attach the FINet industrial source input.'
    PROJECT.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archives[0]) as z:
        assert z.testzip() is None, 'Source ZIP failed CRC.'
        for m in z.infolist():
            assert (PROJECT/m.filename).resolve().is_relative_to(PROJECT.resolve()), 'Unsafe source path'
        z.extractall(PROJECT)
expected_runner_sha256='74f95c6c7ab67fdee4a557a8327629bcbd3c02bfea160fe3b40e04062766ec08'
assert hashlib.sha256((PROJECT/'scripts/run_industrial_kaggle.py').read_bytes()).hexdigest()==expected_runner_sha256, 'Update the dedicated source input; an older notebook output must not replace the verified runner.'
# Kaggle remote-input compatibility: concurrent checks retain every original hash/path check.
import base64, zlib
io_updates=json.loads(zlib.decompress(base64.b64decode('eNrNXH1z27jR/yq4dFqSjkzZTny9UU6dyTm5m0zvkjyJezMdWcOhKEjiiSJZvthWVH33Z3fxQoCkZKXpH70XWwKBxWKx2P3tYunds7KIhnE6r8uqiMMkmIdV6OfbZyP27I7+fV3P44rPWQQ9sg0r65wX93EJLb/9fssj9voNK/MkrgYMqPCcw4+0YtmCVSvObj68YfwRRsQbaPUFxbv0Ll0UQCvKkoRHVZylJYs3eVZU7Car04oXAzbni7BOqnkcVbp3GtVFgXQWdVUXXA+6XRU8nH/MsuTtI4/qKivuUvloFZarJJ7p73+UWaq/ZKUknYcV9lL0PsJX3akI03m20V/LVV3FiRxXwbLUIFjkIogE+3fNHGm9ybcsLFmay0Ef3/2qxrzbhEuuaGVFtALBgjTU4wrmLhdZsSm7ffzmoaaGU+dZEqJEf8vmXAsa9hj7x2mcLmmH1ZCf373n1a188gYelBw2chEnPEDJDdgmTOMFLyv6ivRuXt++/eXDp3dvP7Mxc51ZVlUJdwbMicKZ+pCXtfpY5LzCT8sinuPvVfiFJ2lNbQkHOfPCGdylzPrH2fAKVFF2y+Mkwd9lVPAH/ADyJ+pVllWrWVGXK/qG8ohBRwv89pBlNN+XOIeNcby79M2H316/ew9MOz+HEfTasv+rQ9DbLcitzIUaOnfpx08fbj/cfPgVOwqVDxqVDzb3FY+C+0sHRXEHqr5gIZ6PICviZZyGiVsAU95IrAg/Ax1UJ9Hug9JmyT13PdmBR1kxL6HPZCpaYENZFFZ8iQzGKWvEPWqkFC9YCpSJJhs2/YckhTh18NMSJeD5cRnM48L1RraQoVvJ2e9hUvO3RZEV7sL5LS5L0AKmliLPt6I+Yjv1ce94DbWa5BKWa1wGqI9rPMPVKHIBWQlckyu5xE0D1XLavMV4KogaqCifuz3rtIl6/jLJZq5zNjzz83TpeF6LoJCXoDtqa9tpwmgWP9zZs+8lYd9pTYurp0cB2hdc+UEOQJFAA2HJTX8/D8nWpeGGdwfAklqCHY/17oPFUhS/G0tN6Jm0d+nOP1I02BGafEEjvueCLVwBGvVssYgj8BUg1GITJudZmmyZMi/CGXRkgf+gkghRjFmP6i4LMJ7zoCrqaoUaLFcwZAtnZ4ilrPhmTwpHe92d5iGGGciy+hm4I7cZ66EdLrO6iPgBcYiHfpKFc9c7ILHlDPhPcz8sw6IIt64cA+7pnheV63z65aeOBmr68RdOik1D8Fu3XxUWS16JOb7wIitdFztOLqcDGj+5mHrgH6ttzsfQpY7T6gevV0NO1AF5PPT2oM1AF9AxGieeGTkt0gOjocnunQNCaW+YHvH0fp24Z3KVhthRKPj7CN3Dq9yEVbSC80GaNUR22RwBDvrlEpbcaNzBNVsb7R7Rpl9Bl9jf2IUHPXDLXb3l7IxdXV8f3VIxgx+m26/dy9cpjA/RCIQzccylDQAcAPaF8U0OnvPkLW6chB/O580ONw6xZ2AVp1v7qAklQUgjvov1kVuFvXTdFz+8HDD4AedDdP3Ey3ADBild+u/fvv709vNt30TUhwfxHGZbOMLHN/Y+aNv7YCc0HD607VKPOZI+3g9zxMbuzgEgNGqmJJQkZoJm9RFaxRwB7jk8EN8Gx7WVsBM6fGW39cDGBgBleznQx244ZRKhpTDUUUGARZjEAk8N8RQccek9D6pMYqGwDPKsjB9d75QZNW3UHSBtqtBhyn1WkPEEtP19lvKvmlespVyFV9ffw/waI5v+5asX0kfPMH/fzH0eP/KkmURGQ75ocCHkI9fi+TyNIF4AeT1H/+ZX2WwL0AxOpb/ij/MYQEt12jYhueBhBXMlgMUFddAMgEJc+ndiqQzSUCoo2DJXHmRh58p6AxOfMlmXKpx9TRKth0Fwbxz88D6ME4xXNHDd2daIsJtCbe5TYKWDP/cWVm/PBnvZ2MOncfk/0jyMC7DEYs5zmpMJUsBdY6g06II5E/Cj0vJ4OOH1i+uXxlQ907xVqE9D3kWdJDTwXJh+ddQhQERG2M6cpJm94BCcp8ruNYGSEUYBtK1iivvVeIA2nM/Hl1cvXl5/r3yVSBf89WJ4eQ3/sdnWsI+aS4F+DciLFvMVCSuHJtRreFRDLxxc6hwETkCdOJp9I93gouZ6TTgmOERRS14NOUoCE/FkYp+36VQZffFY0oRThxH40TkF9zClnMC/x60qLS8OKgUqj7rrFpPGiQBKhK8tGy8aTZcyFTpe6ElsncWtFROQ+lyCvKltV/hwUN1+K9YhKcY+peLOm/YusQhgbwkxoFxVDB+BcihACerNK5aCvpSYXIIhoAAFv4/5gxV3CEFPIJ6p3LjihVqQp/dFBpggGnDK0xb3ntbnEnYJ5SziK3DaJRiZCcjUgW2J58SR2YpRrf6+b3aV5h8I6nR25fyCUR943KDNNc8pwOmxTEH5n+gXHNUdnpX9v3dEz8JcgrSPdE0sDGTA5teLBYB60cV4+Cf2Woiq3ggovuAPvIBIL8Rwr+DcSuwJ4AKwvAq3IP60AtME4d+KJ2A56srvaJGcj/3IXrQVgQQ7acQ19WGrcF8w5u05ByW10UO9STZJgM1g92sjqiLGxmAuH93LAdvEYLHQeLkma2fMv7wGN2U1nrNLjw2H7MprJTRktrOsZyUX+Qy3rQdqjycjnB7DNbdZpX6Iz0ZXZ7LLU/7ONbRPkxCjR1PP65ct8XpErHIRh8QqTfluzbcjpaxkicRJh+YBo+/COuKUSov3jeHPC465DCOP5mrzga4Vk70lbBvJb2Cc+XGT/er1D4epqKyb1cNIvw3EY2NEJzdX8kT4wzGrwC5xZSwa9rzG3WKwpQegMOk74B/Zhnk4IWu3WZN3zCE7N6ssg6YQaKUi2pKbJVPr4BC1D2x40gZQJ25lwsUUDuAXUoxAdfIxLe7o1VhjQXnAN9mupwzvSTAYcwfG5YEa6FoUrJ3xLANBhLTzAv7BCoPLENhRyQ4FKjuiEogu+OlJx/IWWceURMOjxAs1epd5vFhwTLIZAhwi4VcsktKHjVyBMZyDlaOkseVg5PEg3rTwjLVq0RE8wSjcfIguiVKzXT34GYIB4r2UC/kIyglnhzNCTmECFleeKvCa4Dte4YJQV/iDtdOYgRTZbQaLnnFEzMh2scWQWK0F6AVNKrqV0LaPUCdvXfhRlm9dw/Wb1EAeFjYhmyP3VWa8GyyIibGnMaKBkpTJcp13myVaReeXWyub7Npaj/09f7NGqYvsajm+LWo+YLRRQbamrydiPwGKOyeLWMGs5d2znYKFiC72GBoo4NnkrYCAJWEYqUa1wulpMxT2r6X7WekncbqW6aOBYM5QVf4Y8bxiHz6TQrUGizst2kjK+x0i8hDP8TSveLxcISBSjKqgb2qd7c4yKHKftuZuJ/+eFIagQqlBSj91/eYM+hXbVl6NEsb/QVYNw+3WHJp+k6AVIhkICR3Lzpo58JbigO6eojbtHJjgBqJcsEJG7sC0UkjNr3MwFFxkK8b008qcWK7QSM0Qx+P+bMuhMYegjJlCGbcSKHKm7tNmSS3PRoi8jFZ8EwLQBnDnAKqn+AFhGCBlvAsZMXHvh/eIRVZlUZZAm7rkw1tF9Cgjsiw9fJueadRyS3QlmVF2zgGbhw7Z6K3gb+PFxmPsQukbZ55FNd6Ig/kWXt3pm1xdtAQznkYr2O91wBM4BnjROmI/h0AKmdhkax7gJYxu7KElzrS4oMa17JrM3cjKFuzNRF6VVcr+lEEoKgFkf8PMe73MKxrK+jdzU8Qrb/rt2LXXi/QmgySCyTPYcVy4o9MEaFGAgCI6VIyc04ihCILPRZagmx9QEfcrFaf/+AJ5aQKc4zullCwA4ECa8UFdluk8OgYCZiiLQRRADFlfoe7RfHa7Ap/PNsBfgoEwDmBREsab5gIO4AlQ3J5r/ZAgvPTZB4UNwyJa4RVewfFAYLHApq4wGyXuJ43QzrzOa4o2yv4Vg7h4WtISb27YT/88f39z/vk1e+lfvGKrqsrL0XD48PDgUzYdDO9miBAGmTmveAgspcvhXBQclEPqdB7Oe2eSWgAzaRxwdtagg71tFSbOAkjzIi/A8oJCje0SBo1MpTGRgRs34Uari42GHwo4A0GFOQUEzf683uTNEFEEk1bjK0ylOnd3eMtNyVVgauzU1eL8h1aeTA1t4qWjPI06KF0kZLR1o5yLrmKALbb79dhIGiHM5NGI5D0oYIhHJiHc2S0IMlC2ntTpQMZmp9TOStwQ0+NdJxsju+2t5GYsE5um6TrK/RudXBIpDPbujWYOrD8F1gPxCRWF63KGgVnVIMNeo4xBBuTdFEAT8ZuMIQa1pADUpp3SDtFL5txwemrw9IrFVwqM9EP2F70OMwSVD38co3yfjpgw0hw09nDA6nSdZg8USmQAnhJQoL6IygqMtABxM+2caGdbZYxAW07RgWLZTkhKkn8xd+jkxKKA6pgcQjuX8HBdQjBUPQCpzkLsOg4lUQWfBGut54Ib1UV8a3IDelO+a29Bl+HPFJzOM1gpmXuUOOPwY8uk9xVrIR6VvTgYhYvg26xIMrATWS8cWrqUCyGI5WMhnbBuHavlHbA8pr0lBT1gcI8u/F07QDeoshiTpXS+9N6cYLefrL4iWwtebrGVSEPl6Ud2wk+fdxGHwcmmGKSdcpOo3rUDFzzenZlbp10VXXRuMvHQNUhY3Al+p+OuhYOJub0+WqdVNhmSpuEsWoUg6jlAQkn37lk8v3s21YnlP7G/h8tlgqF2XlfEUAnnETDFCqIeLHw8xzYGzIPSbH32d85zqbXI8xC5h2l4tB4oigAIwAaDMUZsRY9KIz0BfhG2f0BpEzit99JmI2b49P4XMElVVmMFhq9uVMJCZOrMIkz3dM9DQWi3ihQ06jF4yIo1L8rx5RXFnDk8b+kHzTdgASoJTwHXF2gGsKO/CXPXUrEB05iZXXYK31r3dn+DiOHi4oLE4NIs7M/sGhoglLhgamb8Zg7rq/Sgc2QClp0D+wMHduQYhgP68CXgmy3C2CXoLLkxSTegdcSI/8WCn76gZVijiQGEhnAmnxR8gU3GdnvfzuWG7qHPIqnLlZmW6YVNAEnKkjWqLUtZ3b76ViUfPP4BrBmiiADDtIUwlYMGZhIiCOslwuAxRVZWEhSBj+v5msbTo81LF53Sq3PgmYcb9umXnyQGj7/IXPg/b2Y3BXtzc0s6sCj4v2o8YqjxFaY2o9I3aTYCIBQ/AnLi0gkOBtbBwvcYY8aoyHJcbaWzhIAc4ecfcQUbYJF8L4IW/ijvfTYwAV3+yBIg5IviLRFXyQ5o1WZ0jZfj/Wr4EG5902kmC7/4/eckzinfSCJqPV5Wga5wxhS8rnb2b8B0ZCV3J7YCGj0+dcuBYrM2etyplNbVQQOTzm12CzEO2AFvarhbZYqCB0r6lG34KcJc4FlFuD3Xs1aRjnkVq2gpO/n2XzWGjHiK4BBns3AWU80ybqauR7irLy7mf5W6M9SBprzUGwjzVvA8CSMu6vAldQgxAT3AyQHYA0JHR1un0h+8oluGElOsAjAX9BhxPnL7mnb+J986lJNL/wILJUgCk6fX3UWDYJCfDdgzhM7lUABordPWawmftynwByeAqciHAmuAvqD4xDs4vkX8KF4UgICFpeiRRDwNzimGMC3iJ7yR0PciQr5tnvIcv7deMOgv/1dvCmBtTWUV57dewFDUVe6qyVe1lW/QnzIf9AGk/vl04a5+i0A9usnSRYxeeAm+YIm0NhweRBQm0ZkV4Y5sbU6IFLsy+/rq6Olyd/OFAlHv3wZjssiD+qqCNuiJFZkdRKYSHI+kYKjV7tVFn3tEZajwWGH9n1n51xT6IZnTy/sUwYPVfZ0aPFEKf0rVU29h4NN1f19fLWYs4kDRmpbb0UK1/ROFQXTMDTUWmT2wV0U2ryPMdgYKLgZgdAKyawRFejSro3tfe8uk7sRK8M6VNe7QQEoLF2hd8Gpbw7CDhL476Z5r/LIv3/2fpLhPS6CdEFHJpUwQxTW9JmvpwNZfmRWZokgn198DEr7C/w2bgOEyEGsm6SuAkuzsDmSQ+/MLQHmP8+46lmbf0kidA1HFhTIzTDqIOQS8zCj4H5TeP10TmyInYOLsTD6dXGDhENkOR/dw9hZNZWP082/UbSqMWoutG9y35C3LOJqLPGSxEeM93WecSsFYkhj8v6bZsyJb85Tu5IQ/b3fQl7Xmku0F9yzUPiSCUvN6x1FKazITNKvF40S0Tf2Cb7J77hoz2v3EPLrUrduPUKHAIj7lCUq3SRTgbRy41LFDyTLLs/aJU8zptU7QLExCAFlgsQWADDCxhTcjAaAwjImpiDXIigBB6ClHiD3vtk1GlxfSdAicbFdWLpIstMo5AQ+LC2yQ+Zc4N6zuIWBvrp6mOAnaPh/LiaxDFyaJKKHC8yOlD1tUZI/upTCjuqCKptKGry1cCr+kBNEibUKIthoWFCrTUhXhmnlp/QKs7osr87aaZAVNFsd9WM/lIk9Mv9Qd+izLEs+bOCB27pBxv2ypfLe72WHycvTDgMEPvLXBiP9kNuR7IObUF08PVm/H6AoAtXyDYJzV1lKK7IHw687EdYYraaEwUV1ydnYC8/sWQjtxBnkLfHyKnm3ayyNT1puNqGbowHwXF9vSBdF74rS1TV/yHtwBPZKHaQA+SlW1mJvmX7eU3IhTwmAGMBAjEtD3Amvp6SUsrd+HX+RuJ4MDerstCIy0rHzZspxcSrGEdoaccjtDR85vvi8/CyFyjlMuSujMBLoCgLOTSWFhKsjoMCkpy5mfZzmGSqJ/kISb2Tx0yCX7l6LY7EgXtTGiS1ZXeV3hS8KO90qRNhtt84UEZqKpHSm6Z2ffLDLPV+7F/So31Vx/Gra6w6C8cNa39OPm4l5epTelFo7NSpP9uedzI1Mq3ZH13vfxd+4Pv11/En7EiWQN24mFaoQwQP7WJcbROwzaOoiEWxcCwQ7J7P1ZnDrdcfJqXLw55Jo1TM9xWPOSkdcuVRYXJdAHDT+R6r70bPR67jS3ieP2/ciJyLIHWPbhSoUNusEzGNyjSPvbyhLMfThSq3tCRYJnB9/iSrBbcyCtrdJqmesvA7rFEfYSQUYuk9ZloMFSkS7dapPLSt9Nlq75NsdDaZ2Gw3/qQ94q2n9sQ4+r6FLGSE/9mkVrawj9TQyrpS+hhtWC2bxOeCO4gaklh461WtnQQTLKGOrC67Gk6mstlDlggHmAKBNglrJdA+LbvHgE0xCtwX3hQOtqVL1togtwqYdZhAmqgYRbR5emVGgfx9Ah6lYriwmkdja75YNdDKuqcMWCBo5ekTOQvCrd3FYrOEkFXtCR8APrLRXAMng7QUX29BhbsEsgmwUVSozKkQC9msF2N+l2pJSP33QbfzWB/Fyj5QYllZQncVG3phBtbWQPeu4KzSSHbUVtZjuCkFBfSk1cmZBkOGb33V4pDbRoFfGoqknjnpaW6ArxOsXoStD4HafGolisHA3E7LLz5XTQ9LycypJ18exq1KIEDQdNh8iIAEYDtFYUdV4FeC8s/qCAOk0dR3kg/31ov/8rh9g19GXYs92T88vpxJHlxpZzmznyjuS0UFpClHdmdVS5ZpqG+U7fyQr+pPyxc8DLKMw5WvE0QCYFakazTV7nv7sf3wJ7vtUg11UZzzE/0zyXbQZaUS32XqqxEhy2X6BpaYRdR+34/tCcZ3CgxFn28bxvRgUnuvuv1UhZIfWNKvls//8oV1FN')))
io_prior={'src/industrial_data.py': '694d5d866488fe51899efb2af5712c98591abf49de2dfd31ce69ea01f13e1cf5', 'tests/test_industrial.py': '9c279ae7f51e9e9cb4dacdd44b46cbad0947af6746e21f2c94420182c2b40361'}
io_expected={'src/industrial_data.py': '62db09839658547583e3e8c5dd42f82318e6bed0f9938117d71f4bb332421fa1', 'tests/test_industrial.py': '5656ab3459240d40d32d86377a9ae3afa24dcb2651384b317aa62565b3803235'}
for relative,content in io_updates.items():
    target=PROJECT/relative
    assert hashlib.sha256(target.read_bytes()).hexdigest() in (io_prior[relative],io_expected[relative]), relative
    assert hashlib.sha256(content.encode()).hexdigest()==io_expected[relative], relative
    target.write_text(content,encoding='utf-8')
print('INDUSTRIAL_INTEGRITY_IO_UPDATE',json.dumps(io_expected),flush=True)
os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT))
import torch, torchvision
assert torch.cuda.is_available(), 'Enable a permitted GPU; CUDA is not available.'
print(json.dumps({'torch':torch.__version__,'torchvision':torchvision.__version__,'cuda':torch.version.cuda,'gpu':torch.cuda.get_device_name(0),'free_total_bytes':torch.cuda.mem_get_info(0)}))
# Retain the installed matching GPU torch/torchvision pair. Do not replace CUDA torch.
subprocess.run([sys.executable,'-m','pip','install','torch-dct==0.1.6','timm==1.0.31','pytest','scipy'],check=True)
subprocess.run([sys.executable,'-m','pytest','tests/test_industrial.py','tests/test_domains.py','tests/test_boundary_loss.py','tests/test_training.py','tests/test_industrial_pipeline.py','-q'],check=True)


In [ ]:
# All 15 categories are audited; source image hashes and exact custom splits persist.
# This cell restores verified state and probes a real GPU recovery update.
prepared_caches=list(Path('/kaggle/input').rglob('MVTecAD_supervised_v1/split_manifest.json'))
assert prepared_caches, 'Attach the separate frozen prepared-cache input; saved runs auto-advance old notebook-output inputs.'
print('ATTACHED_PREPARED_CACHE',*[str(p) for p in prepared_caches],flush=True)
subprocess.run([sys.executable,'-u','scripts/run_industrial_kaggle.py'],check=True)
STATE = Path('/kaggle/working/FINetIndustrial')
# Fresh real-batch probe of the restored checkpoint; engineering update only.
import base64, zlib
probe_source=zlib.decompress(base64.b64decode('eNqlWFmP47gRfvevYPqFcketvrtnnOhhsMkuBgE2g8VuXhoGQUuUzVgWtSTVPc5g/nuqeEjy2YvEQLfNo4p1flXkxcXFPxtBtOD11YLbYgU/C/Uq9Ja0Wi3EX0gjYEQ2qpSVFIbYlSCqqmQheU1Eq4CiWIli3SrZ2Ozi4mIykZtWaUu4XrZcGxHH/zaqmVRabUjL7aqWCxIWvsAwbjJbM5nAvwz3ZLIxQtvkJiXG6gT3JYxVshaMTTMtjKpfRTKFvVo01rzczqfT/nirdLGa+ANNoWVrTbbmyyUQb+RScytVEyUQv3e8ZsZyK3YJdNcw2ZQdHA/6Mk8fqQa9mebNOlDqIhtRWM1lI5tlpPncL/2gmkouU7LhazE+o1a8FNoM3PZZJAVvbacF0w2Qr7gu3ziOBK4GfuAuUTPVWrmR/xE6Ba8aMIhgg8zphJz8GP66s5UYIUqGgbC1K4nHOqHAYqJNyZuWVjB07/RQalZyy6PoznUrblYoZiMrEMoNJ5NJKSofcYlzQ0pK8SoLkdOiK/nshsIEMGJaKZv/rBrYoDrbdnGwgcM23Ya5iMw/Po21E19bUVhQwKx4Tu8e7hbPxaJ6vru5KR9uHx8fbj48L4qPi0d+f8ufPghR3Hy4LYuH6uNClNXjU8VvH++fH3h1/1zc8ls6nTneO2KS3EWxl32a+uDL/Friv6aOSlaBIrPbFshy4hSkhDclaVQI2wznMmkYf+Wy5osaonzWawSWNYL80jXgXfF3rZVO6CeXwuSH3/72ifz05TciDUz83kktSlIp7dL2Hz56IRS6jfDGzqgXS2m5lA0wyL1e1xSNpoE/5NX1ghtRy0Zc19zYrLUr6ogWAjij6r1bk8jHc235FqMZdnitcNBvwRBoIdoLl4ng57YDJ78JuVxZw1RTb/NfdRfM5umNsKwBJ9sVKFuaJPB/oYXLJTp/QS5xmc49baE2bS3A/yDHXspGDr1vhr1/3Q0psBAk3sAp31tG9wV7/CnfibgDx/2L113vNms5UKN3AAJEK+BfY+stgVRDvC09xF59fPJ+QTE4QPIbQPIrr2Xpp6MbY1KBppiOzuAmSYJLTVtLy+KeDHdQhFFeMiu+2mTam2EnOZM4mqJqcfBCK8huoVsNxqTzc0r+EitKL96IlFQQ4aKMGvDCSsAeBxr5kPKo9xGN3HoMoWMKvdCeBZ2HaMBQAd77SJxcXn67vDwIqZRQn7B0hjUo5PI58PSf0cGOcKQY4MOeL/yWc276Ph1JnznXw+bEz+6hc+J3hZDASgBgOZQCA0lQdjVEUH60WATyCG0pQYMwrMf5bnovRSOgjirdJ/hPcQaKMijgiirIFjhm+NuTHpajMS6cFjkdDk1J0VfQEJ+O9aiWJ45T5n6zUhYQESnpPezVHhYjXowZ9DKcYtJveIdRr8EpRv2GI4yg0LO12Bqwc0JbcDJEREoBCdstfDvTw7dvHOCHKyjT06VG1JCg7zM6VOLbejbqPZLeFdOX9dxVmTWgWC/t95QAQa8gTjs2ANNHt/vz/OEpYTE6D1ujZN/xo6jwTHwnmwNQAgpAc6ITTxoQDtsWjNm+h0kOQ86xGHLA4fGWdS2mnXGV6RQGLFTXlFxvWc03i5LnIfj3ptNh31roRtQH+/z0KIszAXkfUl420PtgQLw4QV/kPLMqopMzrkTjQut8l95PPfS9SbsKidooBi1wOe4qYs1Cw3RQ5pJXhPAMCmriGboxMnXCJJdehKFo1KJJIhNXKh5ctWq2SeCXQTVshVuKiOCEZwZMnt6m9x8e8G+KZEMrJA1UC/Bh8jrNeF1HYVCQ/rRz1edz4wtlpTp9tYLaQH78/LOwwz3H95F9J+RGsZ8LIzgwgHN/daDXFfWdFHOdlO8DvvUtwuzmvvweeFospQigwNZzhP6qAbQTEFUQ/qMLVN9c7bXgSc/jf8fH4wHbSzwCo2WtFi7rRUvnf74drSxQjFB+wApQpgTWSLROZ3JqNgpSFmWN5oTWBkRxX+P7iJ/4P0pRb4/ROexsRRpc4Vup/FgVGpn5D4r+fitwRL4jziEQ3OqNOQuOOt+daoRM9grIYYk7QcdO1bLj86e4nCpkx+ffrSF7djlXSQbX/eFi4g02gs3zEAg9WufuQO8AoGO7B4BjHT2fdMCnCJFHLkqIhv7icP6C9yteEYyqOUK0Q5z4JlPuvsmMwITGZstdvnPyjUKTCN36jI6K6g6ItdwYUVJsUF1Gw9Y4tRfm0LCoViCrKNQI0Yivk2iBou5KvIPiu8Deu9FKYvptD1mHvCxZj00eXUGcEVoN28ZwNTsOYgdnYGvguY6YItT5lVpw7Z4vNAbZbIjuJdxBa0CTA4ZRuRGa4PXv7vGJdU2x4nDjKenMu9rZF4V1Dyj7nMRXCKB9KBwSD/OGBd3p7FgnEji4hBlJA3dUhlWQ+RoEPNBQJ1hgqzg+CfYd7SYP4wIKF3gukruMAMdhd3C0F/Rs5ocGjS9bdLb3xtVfwgjdV9EbnM6GRBvqxHePAP1jVRJLsefqL1ppyJbwgIE31MRdPEtokk0SFlNS1Z1ZjWBaC4C0JhBPJhNIdsYaDonFnK0YVEpoNhmdhZcRbVzVi4+k2Se97PC15YtbARX9EyQ+jTBWqoKx6Ygy42XJeCBJ6NWVtyYWyop3tc3ptX+rvH5Teg1xfO2anuHOS89yCzfeEbvwCneeCu62V+7Ge3abN3u87+sldrFhq/vCzSaJDsAXQZzIwnub+x3bAT+IF+0w9vynk/8Cu46+NQ==')).decode('utf-8')
assert hashlib.sha256(probe_source.encode()).hexdigest()=='76bd879448e94cda02fe6ea2e35df58bba7cdb6def895d15000afbfbf7e305ca'
probe_path=PROJECT/'scripts/probe_industrial_resume.py'
probe_path.write_text(probe_source,encoding='utf-8')
subprocess.run([sys.executable,'-u',str(probe_path),'--state',str(STATE)],check=True)
for variant in ('baseline','boundary'):
    r=json.loads((STATE/'smoke'/variant/'smoke_report.json').read_text())
    print(variant, json.dumps({'status':r['status'],'gpu':r['hardware'],'losses':[s['loss'] for s in r['steps']],'steady_step_seconds':r['measured_steady_step_seconds'],'steps_per_epoch':r['training_steps_per_epoch'],'full_training_started':False}))


In [ ]:
# Enable only after checking actual GPU, finite gradients, weight updates and recovery above.
RUN_LONG_TRAINING = False
if RUN_LONG_TRAINING:
    assert torch.cuda.is_available()
    checkpoint=STATE/'experiments/baseline/last.pth'
    saved=torch.load(checkpoint,map_location='cpu',weights_only=True)
    assert saved['completed_epochs']>=96, 'Attach epoch 96 or newer; do not repeat completed industrial training.'
    if saved['completed_epochs']==96:
        assert hashlib.sha256(checkpoint.read_bytes()).hexdigest()=='242b7cbf7200d41554087bc9b5a31a68eec081dc4f9bedf56fa15374af37c1a1', 'Recovery differs from the independent epoch-96 backup.'
    print('APPROVED_INDUSTRIAL_RESUME',saved['completed_epochs'],'next_epoch',saved['completed_epochs']+1,'global_step',saved['global_step'],'learning_rate',saved['scheduler_state_dict']['current_lr'],flush=True)
    # Runtime limit only: 104 remaining epochs measured about 7.38 hours, plus setup.
    subprocess.run([sys.executable,'-u','scripts/run_industrial_kaggle.py','--train','--max-hours','8'],check=True)
else:
    print('Long training is disabled. Smoke reports and industrial_recovery.zip are genuine engineering evidence only.')


In [ ]:
# Final evaluation is gated on both complete matched experiments, never partial-vs-full.
paths=[STATE/'experiments'/v/'last.pth' for v in ('baseline','boundary')]
complete=all(p.is_file() and (lambda x:x['completed_epochs']==x['config']['epochs'])(torch.load(p,map_location='cpu',weights_only=True)) for p in paths)
if complete and RUN_LONG_TRAINING:
    from evaluate_industrial import evaluate
    data_root=json.loads((STATE/'data_location.json').read_text())['data_root']
    result=evaluate(data_root,STATE/'split_manifest.json',*paths,STATE/'evaluation',device='cuda:0')
    print(json.dumps(result,indent=2))
    from scripts.run_industrial_kaggle import recovery_zip
    print(recovery_zip(STATE,STATE.parent/'industrial_recovery.zip'))
else:
    print('Full-budget industrial comparison pending. Reattach this saved version output on the next session to resume, not restart.')
